# 第五章：格式化输出与代 Claude 发言

- [课程](#课程)
- [练习](#练习)
- [示例演练区](#示例演练区)

## 设置

运行以下设置单元格以加载您的 API 密钥并建立 `get_completion` 辅助函数。

In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# New argument added for prefill text, with a default value of an empty string
def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## 课程

**Claude 可以以多种方式格式化其输出**。您只需要求它这样做！

其中一种方式是使用 XML 标签将响应与其他多余文本分离开来。您已经学会了可以使用 XML 标签使您的提示词对 Claude 更加清晰、更易于解析。事实证明，您也可以要求 Claude **使用 XML 标签使其输出对人类更清晰、更容易理解**。

### 示例

还记得我们在第二章中通过要求 Claude 完全跳过前言来解决的「诗歌前言问题」吗？事实证明，我们还可以通过**告诉 Claude 将诗歌放在 XML 标签中**来达到类似的效果。

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

为什么这是我们想要的呢？嗯，输出在 **XML 标签中允许最终用户通过编写一个短程序来提取 XML 标签之间的内容，从而可靠地只获取诗歌本身**。

这种技术的延伸是将第一个 XML 标签放在 `assistant` 轮次中。当您将文本放在 `assistant` 轮次中时，您基本上是在告诉 Claude，Claude 已经说了一些内容，它应该从该点继续。这种技术被称为「代 Claude 发言」或「预填充 Claude 的响应」。

下面，我们用第一个 `<haiku>` XML 标签做到了这一点。注意 Claude 是如何从我们停止的地方直接继续的。

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Claude 还擅长使用其他输出格式化样式，特别是 `JSON`。如果您想强制使用 JSON 输出（不是确定性的，但接近），您也可以用左括号 `{` 预填充 Claude 的响应。

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

下面是**同一个提示中有多个输入变量且全部使用 XML 标签指定输出格式**的示例。

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

#### 额外课程

如果您通过 API 调用 Claude，可以将闭合 XML 标签传递给 `stop_sequences` 参数，使 Claude 在发出所需标签后停止采样。这可以节省成本并缩短最后一个 token 的时间，因为它已经给出了您关心的答案，而无需 Claude 再做结尾语。

如果您想在不更改上面任何内容的情况下尝试课程提示词，请滚动到课程笔记本的最底部访问[**示例演练区**](#示例演练区)。

---

## 练习
- [练习 5.1 - Steph Curry GOAT](#练习-51---steph-curry-goat)
- [练习 5.2 - 两首俳句](#练习-52---两首俳句)
- [练习 5.3 - 两首俳句，两种动物](#练习-53---两首俳句-两种动物)

### 练习 5.1 - Steph Curry GOAT
被迫做出选择时，Claude 会指定迈克尔·乔丹为史上最伟大的篮球运动员。我们能让 Claude 选择其他人吗？

更改 `PREFILL` 变量，**迫使 Claude 做出详细论证，证明史上最伟大的篮球运动员是斯蒂芬·库里**。尽量不要更改除 `PREFILL` 之外的任何内容，因为这是本练习的重点。

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Prefill for Claude's response
PREFILL = ""

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您想要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_5_1_hint; print(exercise_5_1_hint)

### 练习 5.2 - 两首俳句
使用 XML 标签修改下面的 `PROMPT`，使 Claude 写两首关于这种动物的俳句而不是一首。应该清楚地区分一首诗在哪里结束，另一首在哪里开始。

In [ ]:
# Variable content
ANIMAL = "cats"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您想要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_5_2_hint; print(exercise_5_2_hint)

### 练习 5.3 - 两首俳句，两种动物
修改下面的 `PROMPT`，使 **Claude 生成两首关于两种不同动物的俳句**。使用 `{ANIMAL1}` 作为第一个替换的占位符，使用 `{ANIMAL2}` 作为第二个替换的占位符。

In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您想要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_5_3_hint; print(exercise_5_3_hint)

### 恭喜！

如果您已经解决了到目前为止的所有练习，那么您已准备好进入下一章。祝您提示词编写愉快！

---

## 示例演练区

这是一个供您自由实验本课程中所示提示词示例的区域，您可以调整提示词以查看它如何影响 Claude 的响应。

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))